# Load Things

In [1]:
import time

In [2]:
# library.py
import pandas as pd
import unicodedata

# diacritics as their unicode value
LOTONE = chr(0x0300)
HITONE = chr(0x0301)
RISETONE = chr(0x030C)
MIDTONE1 = chr(0x0304)
MIDTONE2 = chr(0x0305)
TONECHARS = {LOTONE, HITONE, RISETONE,MIDTONE1,MIDTONE2}

UNDERDOT = chr(0x0323)
UNDERLINE = chr(0x0329)
UNDERDIACS = {UNDERDOT, UNDERLINE}

# split characters into letters (with their diacritics)
def get_letters(text):
    try:
        text = unicodedata.normalize('NFD', text)
    except:
        print(text)
        return []
    text = text.replace(UNDERLINE, UNDERDOT)
    letters = []
    i = 0
    while i < len(text):
        curr_letter = ''
        # look for gb
        if ((i+1) < len(text) and ((text[i] == 'g') and (text[i+1] == 'b'))):
            curr_letter = text[i:i+2]
            i+=2

        # check if next char exists and is a diacritic
        elif ((i+1) < len(text)) and ((text[i+1] in TONECHARS) or text[i+1] in UNDERDIACS):
            if ((i+2) < len(text) and ((text[i+2] in TONECHARS) or text[i+2] in UNDERDIACS)):
                curr_letter = text[i:i+3]
                # print(f"{text[i:i+3]}\t{text[max(i-4, 0):i+4]}\t{text}")
                i+=3 # skip next two chars
            else:
                curr_letter = text[i:i+2]
                i+=2 # skip next char
                
        # normal case (the letter is one single char)
        else: 
            curr_letter = text[i]
            i+=1 # go to next char
        
        # add letter to list
        letters.append(curr_letter.lower())
    return letters

# load in data
def load_dataset(filename):
    return pd.read_csv(filename, header=0, index_col=0)

In [3]:
# syllab.py
from enum import Enum
# from helper import library

# define the possible types of letters
class Letters(Enum):
    SP = 0  # space
    C = 1   # consonant
    M = 2   # m (could be syllabic or consonant)
    N = 3   # n (could be syllabic, consonant, or nasal vowel)
    V = 4   # vowel

# helper functions
def ischar(text):
    return text[0].isalpha()

def chartype(text):
    if ischar(text): 
        if text[0] in ['a', 'e', 'i', 'o', 'u']: return Letters.V
        elif text[0] == 'm': return Letters.M
        elif text[0] == 'n': return Letters.N
        else: return Letters.C
    else: return Letters.SP # punctuation counts as spacing

# syllabify a list of letters, assuming len(letters) != 0
def get_next_syll(letters, syllables, print_it=False):    
    # get types of letters
    # get last char's type
    curr_syll = letters[-1]
    type = chartype(curr_syll)

    letters = letters[:-1]
    if type == Letters.SP: curr_syll = 'SP'

    syllables.insert(0, curr_syll)

    return letters, syllables

def syllabify_letters(letters, print_it = False):
    syllables = []
    while (len(letters) > 0):
        if print_it: print('get_next_syll')
        letters, syllables = get_next_syll(letters, syllables)

    return syllables

def syllabify_df(df):
    syllables = []
    for id, row in df.iterrows():
        letters = get_letters(row['sentence'])
        curr_sylls = syllabify_letters(letters)
        syllables.append([id, curr_sylls])
    return syllables

def ngramify_df(df):
    ngrams = []
    for id, row in df.iterrows():
        letters = get_letters(row['sentence'])
        ngrams.append([id, letters])
    return ngrams

In [4]:
# Get syllable and word counts
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    for i in dup_indices.index:
        print(f'{i}: {df.loc[i]}')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped


# Prepare Tests

In [5]:
iroyin_full = load_dataset('/Users/graven2/Documents/THESIS/data/iroyinspeech_full2_deduped.csv')
multidiac_full = load_dataset('/Users/graven2/Documents/THESIS/data/multidiac_full.csv')
yad_full = load_dataset('/Users/graven2/Documents/THESIS/data/yad_full_CLEAN.csv')

In [6]:
print('---Iroyin---')
iroyin_ngrams = syllabify_df(iroyin_full)
print('---MultiDiac---')
multidiac_ngrams = syllabify_df(multidiac_full)
print('---YAD---')
yad_ngrams = syllabify_df(yad_full)

---Iroyin---
---MultiDiac---
---YAD---
nan


In [7]:
all_letters = pd.DataFrame(columns=['Source', 'ID', 'Sentence', 'Syllables'])
print('---IROYIN---')
for row in iroyin_ngrams:
    source = 'IroyinSpeech'
    id = row[0]
    sentence = iroyin_full.loc[id, 'sentence']
    ngrams = row[1]
    all_letters.loc[len(all_letters)] = [source, id, sentence, ngrams]

print('---MultiDiac---')
for row in multidiac_ngrams:
    source = 'MultiDiac'
    id = row[0]
    sentence = multidiac_full.loc[id, 'sentence']
    ngrams = row[1]
    all_letters.loc[len(all_letters)] = [source, id, sentence, ngrams]

print('---YAD---')
for row in yad_ngrams:
    source = 'YAD'
    id = row[0]
    sentence = yad_full.loc[id, 'sentence']
    ngrams = row[1]
    all_letters.loc[len(all_letters)] = [source, id, sentence, ngrams]

---IROYIN---
---MultiDiac---
---YAD---


In [8]:
deduped = dup_rows(all_letters)

Original Length: 43494
48 TOTAL DUPLICATES FOUND
249: Source                                            IroyinSpeech
ID                                                   00250.wav
Sentence     A ti gbé ìgbésẹ̀ láti bá àwọn tí ọ̀rọ̀ náà kàn...
Syllables    [a, SP, t, i, SP, gb, é, SP, ì, gb, é, s, e...
Name: 249, dtype: object
253: Source                                            IroyinSpeech
ID                                                   00254.wav
Sentence     A ó túbọ̀ máa fọwọ́sowọ̀pọ́ pẹ̀lú ìjọba ìpínlẹ...
Syllables    [a, SP, ó, SP, t, ú, b, ọ̀, SP, m, á, a, S...
Name: 253, dtype: object
278: Source                                            IroyinSpeech
ID                                                   00279.wav
Sentence     Ọjọ́ kẹ́rìnlélógún, oṣú kejìlá, ọdún yìí ní il...
Syllables    [ọ, j, ọ́, SP, k, ẹ́, r, ì, n, l, é, l, o...
Name: 278, dtype: object
312: Source                                            IroyinSpeech
ID                                   

In [62]:
deduped.to_csv('ngramified.csv')

# Run NGrams

In [9]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [10]:
# ngrams.py
from enum import Enum

# Create consistent labels for tones
class Tones(Enum):
    H = 1
    M = 0
    L = -1

# All the letters that could have an underdot
DOTCONS = ['s']
DOTVOWELS = ['e', 'o']
DOTLETTERS = ['s', 'e', 'o']

# possible vowels in Yoruba
VOWELS = ['a','e','i','o','u']

# remove the diacritics from a syllable
# can keep 'underdiacs', 'tone', or 'none'
def _rm_diacritics_letter(letter, keep):
    # corner cases
    if letter == 'SP': return letter
    if letter == 'ERR': return letter

    # normal syllable
    include = [letter[0]] # keep original letter
    
    # check second char
    if (len(letter) > 1) and ((letter[1] in UNDERDIACS) and (keep=='underdiacs')):
        include.append(letter[1])
    if (len(letter) > 1) and ((letter[1] in TONECHARS) and (keep=='tone')):
        include.append(letter[1])

    # check third char
    if (len(letter) > 2) and ((letter[2] in UNDERDIACS) and (keep=='underdiacs')):
        include.append(letter[2])
    if (len(letter) > 2) and ((letter[2] in TONECHARS) and (keep=='tone')):
        include.append(letter[2])

    # create string from included characters
    return ''.join(include)

# remove diacritics from a set of syllables
def rm_diacritics_row(row, keep):
    return [_rm_diacritics_letter(x, keep) for x in row['Syllables']]

# remove diacritics from df
# can keep 'underdiacs', 'tone', or 'none'
def rm_diacritics_df(df, keep):
    new_df = df.copy()
    new_df['Syllables'] = new_df.apply(lambda row: rm_diacritics_row(row, keep), axis=1)
    return new_df

# determine which letters get dots
# type = both, vowels, or cons
def _dots_present(letter, type):
    if (letter[0] in DOTCONS) and (type=='both' or type=='cons'):
        # print('DOTCONS', letter, len(letter))
        if (len(letter) > 1) and (letter[1] in UNDERDIACS): return 1
        elif (len(letter) > 2) and (letter[2] in UNDERDIACS): return 1
        else: return 0
    if (letter[0] in DOTVOWELS) and (type=='both' or type=='vowels'):
        # print('DOTVOWELS', letter, len(letter))
        if (len(letter) > 1) and (letter[1] in UNDERDIACS): return 1
        elif (len(letter) > 2) and (letter[2] in UNDERDIACS): return 1
        else: return 0
    return 0


# add given dots to a syllable
def _add_dots(letter, dots):
    if (letter[0] not in DOTCONS) and (letter[0] not in DOTVOWELS): return letter
    if (dots == 1): 
        return ''.join([letter, UNDERDOT])
    else: return letter

# find the n-sized context string for syllable i in syllables
def _get_context(letters, n, i, keep):
    context = []
    for j in range(1, n+1):
        # get -Syl
        # insert at FRONT of list
        if (i-j >= 0): 
            curr_context = letters[i-j]
            curr_context = _rm_diacritics_letter(curr_context, keep=keep)
            context.insert(0, curr_context)
        else: context.insert(0, '<') # start of sentence token

        # get +Syl
        # add to END of list
        if (i+j < len(letters)):
            curr_context = letters[i+j]
            curr_context = _rm_diacritics_letter(curr_context, keep=keep)
            context.append(curr_context)
        else: context.append('>') # end of sentence token
    
    # merge context into a string
    context_str = '.'.join(context) # -Syl.-Syl.+Syl.+Syl
    return context_str


# get n-grams
# n is the number of items before and after to consider
def _syll_grams(letters, counts, n, keep):
    if not counts: 
        counts = []
        for i in range(n+1): counts.append(dict())
    
    for i, letter in enumerate(letters):
        letter_str = ''.join(_rm_diacritics_letter(letter, keep=keep))
        # corner case
        if (letter == 'SP') or (letter == 'ERR') or (letter == 'UNK'):
            letter_str = letter

        # counts has format [{syll: {-Syl.-Syl.+Syl.+Syl : {'1 0 0' : count,  '1 1 0' : count, ...}}}, ...]
        # find all contexts in range 0-n (inclusive)
        for j in range(n+1):
            # get contexts for this syllable so far
            poss_contexts = counts[j].get(letter_str, dict())

            # get current context for syllable
            context_str = _get_context(letters, j, i, keep)
            # print(context_str)

            # update with new dots
            # get previous counts
            curr_dots = _dots_present(letter, 'both')
            context_dots = poss_contexts.get(context_str, dict())
            curr_dot_count = context_dots.get(curr_dots, 0)

            # update all dictionaries
            context_dots.update({curr_dots : curr_dot_count + 1})
            poss_contexts.update({context_str : context_dots})
            counts[j].update({letter_str : poss_contexts})
            # print(counts[j])
    return counts

# create a full n-gram count from a df
def create_syll_grams(df, n, keep):
    counts = dict()
    for _, row in df.iterrows():
        counts = _syll_grams(row['Syllables'], counts, n, keep=keep)
    return counts

# predict the dots for a syllable
def pred_dots(letters, counts, n, keep, print_it=False):
    with_dots = []

    for i, letter in enumerate(letters):
        # corner case
        if letter == 'SP' or letter == 'ERR':
            with_dots.append(letter)
            continue
        
        letter_str = ''.join(letter)

        # try n, n-1, ..., 0 (and stop as soon as it is possible)
        for j in range(n, -1, -1):
            # get context
            context_str = _get_context(letters, j, i, keep=keep)

            # collect stored counts
            possible_dots = counts[j].get(letter_str, dict()).get(context_str, dict())

            # find max
            pred_dots = 0
            max_use = -1
            for key,value in possible_dots.items():
                if value > max_use:
                    max_use = value
                    pred_dots = key
            syll_with_dots = _add_dots(letter, pred_dots)
            # print(letter_str, context_str, possible_dots, syll_with_dots)


            # break if this syllable/sequence of syllables exists in these n-grams
            if not possible_dots: continue # dictionary is empty, so keep trying
            else: 
                if(print_it) : print('n-gram found, stopping; j = ',j,  letter_str, context_str)
                break # dictionary was found, so stop going to smaller syllables

        with_dots.append(syll_with_dots)

    return with_dots

# do full predictions
def predict_all_dots(df, counts, n):
    new_df = df.copy()
    new_df['Prediction'] = new_df.apply(lambda row: pred_dots(row['Syllables'], counts, n), axis=1)
    return new_df

# calculate word error rate for a row, returns (wrong words, total words)
def _eval_row(row, type='both', print_it=False):
    correct = row['Syllables']
    pred = row['Prediction']

    wrong_words = 0
    total_words = 0
    in_word = False # identifies whether currently in a word or not
    curr_word_accurate = True # identifies whether the current word has gotten a tone wrong yet

    # iterate through syllables
    for i in range(len(correct)):
        # check if tones match
        corr_tone = _dots_present(correct[i], type)
        pred_tone = _dots_present(pred[i], type)
        if print_it: print(corr_tone, pred_tone)
        if corr_tone != pred_tone: 
            curr_word_accurate = False
            if print_it: print('WRONG DOTS', corr_tone, pred_tone)

        # check if a word is finished
        if in_word:
            # word has ended
            if correct[i] == 'SP':
                in_word = False
                if not curr_word_accurate: 
                    wrong_words += 1
                    if print_it: print('WRONG', correct,pred)
                total_words += 1
                curr_word_accurate = True # reset accuracy
        if correct[i] != 'SP': in_word = True
    if in_word:
        if not curr_word_accurate: wrong_words+=1
        total_words+=1

    return pd.Series({'Wrong Words' : wrong_words, 'Total Words' : total_words})

# determine wrong words in df of syllables
def evaluate(df, print_it=False):
    new_df = df.copy()
    new_df[['Wrong Words', 'Total Words']] = new_df.apply(lambda row: _eval_row(row, print_it=print_it), axis=1, result_type='expand')
    return new_df


In [11]:
tester = deduped.loc[4, 'Syllables']
print(tester)

# 29 = ['r', 'ọ̀']
# 2 = ['SP']
# 6 = ['í']
# 22 = ['t', 'u', 'n']
testI = 1
keep = 'tone'
orig_syll = tester[testI]
new_syll = _rm_diacritics_letter(orig_syll, keep=keep)
print(orig_syll, new_syll)

print(_dots_present(orig_syll, 'both'))
print(_dots_present(new_syll, 'both'))

minitester = tester[0:30]
ngram_counts = _syll_grams(minitester, dict(), 4, keep)
print(ngram_counts[2])

untoned = [_rm_diacritics_letter(x, keep) for x in minitester]
print(untoned)
toned = pred_dots(untoned, ngram_counts, 4, keep)
print(toned)

['ọ̀', 'j', 'ọ̀', 'gb', 'ọ́', 'n', 'SP', 'a', 'j', 'á', 'ń', 'l', 'é', 'k', 'o', 'k', 'ò', 'SP', 's', 'ọ̀', 'r', 'ọ̀', 'SP', 'l', 'á', 's', 'ì', 'k', 'ò', 'SP', 'ì', 'p', 'à', 'd', 'é', 'SP', 't', 'í', 'SP', 'ó', 'SP', 'ṣ', 'e', 'SP']
j j
0
0
{'ò': {'<.<.j.ò': {1: 1}, 'ò.j.g.ó': {1: 1}, 'o.k.SP.s': {0: 1}, 'SP.s.r.ò': {1: 1}, 'ò.r.SP.l': {1: 1}, 'ì.k.SP.>': {0: 1}}, 'j': {'<.ò.ò.g': {0: 1}, 'SP.a.á.ń': {0: 1}}, 'g': {'j.ò.ó.n': {0: 1}}, 'ó': {'ò.g.n.SP': {1: 1}}, 'n': {'g.ó.SP.a': {0: 1}}, 'SP': {'ó.n.a.j': {0: 1}, 'k.ò.s.ò': {0: 1}, 'r.ò.l.á': {0: 1}, 'k.ò.>.>': {0: 1}}, 'a': {'n.SP.j.á': {0: 1}}, 'á': {'a.j.ń.l': {0: 1}, 'SP.l.s.ì': {0: 1}}, 'ń': {'j.á.l.é': {0: 1}}, 'l': {'á.ń.é.k': {0: 1}, 'ò.SP.á.s': {0: 1}}, 'é': {'ń.l.k.o': {0: 1}}, 'k': {'l.é.o.k': {0: 1}, 'k.o.ò.SP': {0: 1}, 's.ì.ò.SP': {0: 1}}, 'o': {'é.k.k.ò': {0: 1}}, 's': {'ò.SP.ò.r': {0: 1}, 'l.á.ì.k': {0: 1}}, 'r': {'s.ò.ò.SP': {0: 1}}, 'ì': {'á.s.k.ò':

In [22]:
for n in range(0,4):
    # split data
    curr_df = deduped
    # split into 10 folds, make it even across all three datasets
    skf = StratifiedKFold(n_splits=10,shuffle=False)
    X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
    y = curr_df['Source'].to_numpy() # stratify across the datasets

    # run predictions for all folds
    WERs = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        # if i > 1: break
        # n = 6
        keep = 'none'

        print(f"--- Fold {i} ---")
        train_df = curr_df.iloc[train_index].copy()
        test_df = curr_df.iloc[test_index].copy()

        # create n-grams
        start_time = time.time()
        counts = create_syll_grams(train_df, n=n, keep=keep)
        end_time = time.time()
        print('N-Gram Timing : ', (end_time-start_time), flush=True)

        # create detoned version (to test predictions)
        start_time = time.time()
        detoned = test_df.apply(lambda row: rm_diacritics_row(row, keep=keep), axis=1)

        # predict tones
        test_df['Prediction'] = detoned.apply(lambda row: pred_dots(row, counts, n=n, keep=keep))
        end_time = time.time()
        print('Prediction Timing : ', (end_time - start_time), flush=True)

        # evaluate
        start_time = time.time()
        test_df = evaluate(test_df, print_it=False)
        wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
        end_time = time.time()
        print('Evaluation Timing : ', (end_time - start_time), flush=True)
        print('WER = ', wer, flush=True)
        WERs.append(wer)
        
        # print(test_df)

        # print(f"  Train: index={train_index}")
        # print(f"  Test:  index={test_index}")

    print(float(sum(WERs)) / float(len(WERs)))

--- Fold 0 ---
N-Gram Timing :  4.418260812759399
Prediction Timing :  0.25783705711364746
Evaluation Timing :  0.5508978366851807
WER =  34.13111342351717
--- Fold 1 ---
N-Gram Timing :  3.6314589977264404
Prediction Timing :  0.2784390449523926
Evaluation Timing :  0.4608428478240967
WER =  33.891854188148244
--- Fold 2 ---
N-Gram Timing :  3.732740879058838
Prediction Timing :  0.20075297355651855
Evaluation Timing :  0.4363820552825928
WER =  34.86430912681447
--- Fold 3 ---
N-Gram Timing :  3.630413770675659
Prediction Timing :  0.31922125816345215
Evaluation Timing :  0.45595788955688477
WER =  33.72889339807232
--- Fold 4 ---
N-Gram Timing :  3.634428024291992
Prediction Timing :  0.21916794776916504
Evaluation Timing :  0.4331247806549072
WER =  35.47979335360715
--- Fold 5 ---
N-Gram Timing :  3.5807738304138184
Prediction Timing :  0.2613959312438965
Evaluation Timing :  0.532721996307373
WER =  30.88011214725422
--- Fold 6 ---
N-Gram Timing :  3.5561978816986084
Prediction T